# Lin 2025 rat Müller external validation

Threshold-aware external validation using the authors' Müller supplementary DEG table at 1, 2, and 3 months. Missing genes are treated as censored rather than unchanged.

**Important:** These notebooks document the finalized analysis logic. Public raw data are downloaded from the cited repositories; large intermediate objects and outputs are intentionally not embedded in the GitHub version. Where a dataset lacks condition-level biological replication, results are explicitly treated as descriptive.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import requests, pandas as pd, json

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")
OUT = ROOT/"07_Lin2025_Rat_Muller_Validation"
OUT.mkdir(exist_ok=True)

with open(ROOT/"signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    sig = json.load(f)
print("Early184:", len(sig["early_up_transient_all"]))

j = requests.get("https://api.figshare.com/v2/articles/27273519").json()

files = pd.DataFrame(j["files"])
display(files[["name","size","download_url"]])

In [ ]:
import requests, pandas as pd

xlsx = OUT/"Lin2025_Supplementary_Table_S2.xlsx"
xlsx.write_bytes(requests.get(
    "https://ndownloader.figshare.com/files/49907814"
).content)

xls = pd.ExcelFile(xlsx)

print("Sheets:", xls.sheet_names)

for s in xls.sheet_names:
    df = pd.read_excel(xlsx, sheet_name=s)
    print(f"\n### {s} {df.shape}")
    print(df.columns.tolist())
    display(df.head(3))

In [ ]:
mg = pd.read_excel(xlsx, sheet_name="MG (cluster 4)")

cols = {
    "1M": ("Unnamed: 1",  "p_val",   "avg_log2FC"),
    "2M": ("Unnamed: 6",  "p_val.1", "avg_log2FC.1"),
    "3M": ("Unnamed: 11", "p_val.2", "avg_log2FC.2")
}

rat = {}
for tp,(g,p,l) in cols.items():
    x = mg[[g,p,l]].copy()
    x.columns = ["gene","p","log2FC"]
    x = x.dropna(subset=["gene","log2FC"])
    rat[tp] = x
    print(tp,
          "genes =", len(x),
          "| p range =", (x.p.min(), x.p.max()),
          "| log2FC range =", (x.log2FC.min(), x.log2FC.max()))

In [ ]:
for tp,x in rat.items():
    a = x.log2FC.abs()

    print(
        tp,
        "| n =", len(x),
        "| min |LFC| =", round(a.min(),4),
        "| q10 =", round(a.quantile(.10),4),
        "| median =", round(a.median(),4),
        "| p>0.05 =", (x.p > .05).sum(),
        "| |LFC|<0.25 =", (a < .25).sum()
    )

In [ ]:
!pip -q install gprofiler-official

from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)

early184 = sig["early_up_transient_all"]

rat_orth = gp.orth(
    organism="mmusculus",
    target="rnorvegicus",
    query=early184
)[["incoming","name"]].dropna().drop_duplicates()

rat_orth.columns = ["mouse_gene","rat_gene"]

rows = []

for tp,x in rat.items():
    z = rat_orth.merge(x, left_on="rat_gene", right_on="gene")

    rows.append({
        "timepoint": tp,
        "mapped_Early184": rat_orth.mouse_gene.nunique(),
        "reported_Early184": z.mouse_gene.nunique(),
        "up": (z.log2FC > 0).sum(),
        "down": (z.log2FC < 0).sum(),
        "fraction_up": (z.log2FC > 0).mean(),
        "median_reported_log2FC": z.log2FC.median()
    })

early_rat = pd.DataFrame(rows)
display(early_rat)

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

rows = []

for tp,x in rat.items():
    z = rat_orth.merge(x, left_on="rat_gene", right_on="gene")

    early_genes = set(z["gene"])
    bg = x[~x["gene"].isin(early_genes)]

    a = (z.log2FC > 0).sum()   # Early184 up
    b = (z.log2FC < 0).sum()   # Early184 down
    c = (bg.log2FC > 0).sum()  # background up
    d = (bg.log2FC < 0).sum()  # background down

    OR, p = fisher_exact([[a,b],[c,d]], alternative="two-sided")

    rows.append([tp,a,b,c,d,a/(a+b),c/(c+d),OR,p])

rat_bias = pd.DataFrame(rows, columns=[
    "timepoint","Early_up","Early_down","BG_up","BG_down",
    "Early_fraction_up","BG_fraction_up","odds_ratio","p"
])

rat_bias["FDR_BH"] = multipletests(rat_bias["p"], method="fdr_bh")[1]

display(rat_bias.round(4))

In [ ]:
locked = {
    "Early184": sig["early_up_transient_all"],
    "IFN6": sig["early_IFN_alpha_beta"],
    "Retinoid5": sig["early_retinoid_metabolism"],
    "ECM13": sig["early_ECM_GAG_exploratory"]
}

allgenes = sorted(set(sum(locked.values(), [])))

ro = gp.orth(
    organism="mmusculus",
    target="rnorvegicus",
    query=allgenes
)[["incoming","name"]]

ro.columns = ["mouse_gene","rat_gene"]
ro = ro.dropna().drop_duplicates()

rows = []

for s,genes in locked.items():
    smap = ro[ro.mouse_gene.isin(genes)]

    for tp,x in rat.items():
        z = smap.merge(x, left_on="rat_gene", right_on="gene")
        sg = set(z.gene)
        bgx = x[~x.gene.isin(sg)]

        up,down = (z.log2FC>0).sum(), (z.log2FC<0).sum()
        bu,bd = (bgx.log2FC>0).sum(), (bgx.log2FC<0).sum()

        OR,p = fisher_exact([[up,down],[bu,bd]]) if up+down else (np.nan,np.nan)

        rows.append([
            s,tp,smap.mouse_gene.nunique(),len(z),
            up,down,z.log2FC.median() if len(z) else np.nan,
            up/(up+down) if up+down else np.nan,OR,p
        ])

rat_locked = pd.DataFrame(rows, columns=[
    "signature","timepoint","mapped_n","reported_n","up","down",
    "median_log2FC","fraction_up","odds_ratio","p"
])

rat_locked["FDR_BH"] = multipletests(
    rat_locked["p"].fillna(1), method="fdr_bh"
)[1]

display(rat_locked.round(4))

In [ ]:
rat_orth.to_csv(OUT/"Lin2025_Early184_mouse_rat_orthologs.csv", index=False)
early_rat.to_csv(OUT/"Lin2025_Early184_thresholded_trajectory.csv", index=False)
rat_bias.to_csv(OUT/"Lin2025_Early184_background_bias_tests.csv", index=False)
rat_locked.to_csv(OUT/"Lin2025_all_locked_signatures_thresholded_tests.csv", index=False)

print("LIN2025 RAT MULLER VALIDATION CLOSED")
print("Early184 reported:", dict(zip(early_rat.timepoint, early_rat.reported_Early184)))
print("No locked signature survived FDR correction.")